In [ ]:
import os
import random
import numpy as np
import torch
import torchvision
from sklearn.model_selection import train_test_split

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED=42

set_seed(SEED)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] Available device: {device}")

In [ ]:
BASE_DIR = os.path.dirname(os.getcwd())

DATASET="CIFAR10"   
SPLIT=0.2

DATA=os.path.join(BASE_DIR,"data")

os.makedirs(DATA,exist_ok=True)

In [ ]:
ROOT_DIR=os.path.join(DATASET,f"seed_{SEED}",f"split_{SPLIT}")
os.makedirs(ROOT_DIR,exist_ok=True)


In [ ]:
if DATASET=="CIFAR10":

    train_dataset=torchvision.datasets.CIFAR10(root=DATA,train=True,download=True,transform=None)

elif DATASET=="CIFAR100":

    train_dataset=torchvision.datasets.CIFAR100(root=DATA,train=True,download=True,transform=None)

elif DATASET=="Food101":

    train_dataset=torchvision.datasets.Food101(root=DATA,split="train",download=True,transform=None)

elif DATASET=="Flowers102":

    train_dataset=torchvision.datasets.Flowers102(root=DATA,split="train",download=True,transform=None)
    val_dataset=torchvision.datasets.Flowers102(root=DATA,split="val",download=True,transform=None)

else:
    raise ValueError(f"Unsupported dataset: {DATASET}")


In [ ]:

if DATASET=="Flowers102":
    train_indices=np.arange(len(train_dataset))
    val_indices=np.arange(len(val_dataset))
else:
    labels=[train_dataset[i][1] for i in range(len(train_dataset))]
    all_indices=np.arange(len(train_dataset))
    train_indices,val_indices=train_test_split(
        all_indices,
        test_size=SPLIT,
        stratify=labels,
        random_state=SEED
    )

train_size=len(train_indices)
val_size=len(val_indices)

train_indices_path=os.path.join(ROOT_DIR,"train_indices.npy")
val_indices_path=os.path.join(ROOT_DIR,"val_indices.npy")

np.save(train_indices_path,train_indices)
np.save(val_indices_path,val_indices)

print(f"[OK] Stratified split completed for {DATASET}:\n")

print(f"Train size: {train_size}")
print(f"Validation size: {val_size}")
print(f"Total samples: {len(train_dataset)}\n")

if DATASET=="Flowers102":
    train_labels=[train_dataset[i][1] for i in train_indices]
    val_labels=[val_dataset[i][1] for i in val_indices]
else:
    train_labels=[labels[i] for i in train_indices]
    val_labels=[labels[i] for i in val_indices]

if DATASET=="CIFAR10":
    num_classes=10
elif DATASET=="CIFAR100":
    num_classes=100
elif DATASET=="Food101":
    num_classes=101
elif DATASET=="Flowers102":
    num_classes=102

print(f"\n[INFO] Class distribution verification:\n")
for class_id in range(num_classes):
    train_count=sum(1 for l in train_labels if l==class_id)
    val_count=sum(1 for l in val_labels if l==class_id)
    print(f"  Class {class_id:3d}: Train={train_count:5d} ({train_count/train_size*100:5.1f}%) | Val={val_count:4d} ({val_count/val_size*100:5.1f}%)")

print(f"\n[OK] Train indices saved to: {train_indices_path}\n")
print(f"[OK] Validation indices saved to: {val_indices_path}\n")